# Tutorial 9: How Much Fine-Tuning Is Enough, and What Does It Cost?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/byu-matrix-lab/torchlingo/blob/main/docs/docs/tutorials/09-fine-tuning-nllb.ipynb)

**Reading. About 30 minutes, plus about 15 minutes of training on an A100, L4 or G4; the
training prints its own estimate.**

A14 fine-tunes NLLB-200 on your pairs once, at one budget, and measures what that bought. Any
fine-tuning you do after it, in a Final Project above all, has to answer two more questions
first: **how many steps are enough**, and **what does fine-tuning cost the model** it started
from? This notebook answers both on your own pairs. It fine-tunes NLLB from English into your
language, scores it after 100, 300 and 1,000 steps, looks at what it did to the direction it
was not trained on, and shows what a learning rate set too high does.

By the end you will be able to:

1. read a learning curve, the score after each budget, and say where more steps stop paying
   for themselves;
2. say what fine-tuning one direction did to the model's other abilities (the reverse
   direction, and a third language), and why a model can forget what it was not reminded of;
3. recognize a learning rate set too high, from its losses and from its translations.

These serve Lecture 14's outcome *explain tagging and why zero-shot works at all*: what a
multilingual model keeps, and loses, when it is trained on one pair. They are also the two
questions a Final Project that fine-tunes must answer.

It uses what A14 sets up: your language's NLLB code (A14's Step 1 says how to find it, or which
code to borrow), and the library's fine-tuning, `fine_tune`, whose ingredients A14's Step 4
explains.

**Running in Google Colab?** Runtime → Change runtime type → an **A100, L4 or G4** GPU, not
the T4; then Run all. It stops at the prediction until you write it. It runs on a built-in
English–Spanish sample until you change three lines in Step 1. When you finish, disconnect the
GPU: Runtime → Disconnect and delete runtime. The last cell says why.

<!-- GENERATED:purpose - change the torchlingo metadata, not this cell -->

**TorchLingo tutorial** — assigned reading for Lecture 14 (*Multilingual NMT and "Zero-shot" NMT*). A head start on **A14**.

In [ ]:
# Install TorchLingo in Colab. Nothing to uncomment. Pip's "you may need to restart the
# kernel" does not apply to a first install; if the install fails, the import below stops here.
import sys
if "google.colab" in sys.modules:
    %pip install --quiet "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
    # pip keeps a TorchLingo already in this session when its version number matches, even if
    # it is older code; this swaps in the course branch's, leaving the dependencies alone.
    %pip install --quiet --force-reinstall --no-deps "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
import torchlingo


In [ ]:
# Check the setup and connect Google Drive, where this notebook keeps its checkpoints and output.
from torchlingo.colab import course_dir, setup

env = setup(gpu=True, drive=True)       # stops with instructions if there is no GPU
device = env.device
OUT_DIR = course_dir(env.drive_dir, "tutorial9", create=True)   # MyDrive/CS479/tutorial9

import json
import tempfile
from pathlib import Path

import pandas as pd
import torch
import torchlingo


## Step 1: your pairs, your code, and a third language

Set `A8_SPLIT` to the folder your A8 split is in, and `CODE` to your language's NLLB code, the
one A14 used (borrowed, if NLLB does not cover your language). `THIRD` is a language you can
read, as an NLLB code, other than English and yours: Step 4 asks whether fine-tuning changed
how the model writes it. With `A8_SPLIT` left as `None`, the notebook runs on a built-in
English–Spanish sample.

Every score here is on 200 of your test sentences, decoded greedily, to keep the notebook
quick; A14 scores the whole test set, both ways.

In [ ]:
A8_SPLIT = None     # <- e.g. env.drive_dir / "CS479" / "assignment8"
CODE = "spa_Latn"   # <- your language's NLLB code, as in A14 (borrowed if NLLB lacks it)
THIRD = "fra_Latn"  # <- a third language you can read, e.g. "fra_Latn", "por_Latn", "deu_Latn"

SEED, TEST_N, ENG = 479, 200, "eng_Latn"


In [ ]:
# Setup: run this cell, no need to read it. It loads your split, or the sample, and NLLB.
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

from torchlingo.evaluation import compute_chrf
from torchlingo_harness.nllb import NLLB_MODEL, NLLBTranslator

if A8_SPLIT is None:
    from torchlingo.colab import fetch_data

    (sample,) = fetch_data(["data/example.tsv"])
    pairs = pd.read_csv(sample, sep="\t", dtype=str, keep_default_na=False)
    words = pairs.src.str.split().str.len()
    pairs = pairs[(pairs.kind == "transcript") & words.between(4, 40)][["src", "tgt"]]
    pairs = pairs.drop_duplicates("src").sample(frac=1, random_state=SEED)
    train, val, test = pairs[:4000], pairs[4000:4200], pairs[4200:4400]
    assert CODE == "spa_Latn", "the built-in sample is Spanish; set A8_SPLIT to use your own code"
    print("*** The built-in English-Spanish sample. Set A8_SPLIT and CODE in the cell above. ***\n")
else:
    A8_SPLIT = course_dir(env.drive_dir, Path(A8_SPLIT).name,
                          needs=["train.tsv", "val.tsv", "test.tsv"],
                          hint="The A8 kickoff notebook writes them; run it that far first.")
    train, val, test = (pd.read_csv(Path(A8_SPLIT) / f"{n}.tsv", sep="\t", dtype=str,
                                    keep_default_na=False) for n in ("train", "val", "test"))
test = test.sample(min(TEST_N, len(test)), random_state=SEED)
# One direction only, English into yours: each pair carries its two codes, as A14's do.
train, val = (f[["src", "tgt"]].assign(src_code=ENG, tgt_code=CODE) for f in (train, val))

tokenizer = AutoTokenizer.from_pretrained(NLLB_MODEL)
codes = {t for t in tokenizer.additional_special_tokens if len(t) == 8 and t[3] == "_"}
assert CODE in codes and THIRD in codes and THIRD not in (ENG, CODE), (
    "CODE and THIRD must be NLLB codes, and THIRD neither English nor yours")


def fresh_nllb():
    """NLLB as it comes, in float32 on the GPU."""
    return AutoModelForSeq2SeqLM.from_pretrained(NLLB_MODEL, dtype=torch.float32).to(device)


def translate(model, sentences, src_code, tgt_code):
    tokenizer.src_lang = src_code
    return NLLBTranslator(model.eval(), tokenizer, tgt_code, device, NLLB_MODEL)(sentences, batch_size=32)


def chrf_both_ways(model):
    """chrF into your language, the direction trained, and into English, the one not trained."""
    into_x = compute_chrf(translate(model, list(test.src), ENG, CODE), list(test.tgt)).score
    into_en = compute_chrf(translate(model, list(test.tgt), CODE, ENG), list(test.src)).score
    return round(into_x, 2), round(into_en, 2)


model = fresh_nllb()
print(f"{len(train):,} training pairs; {len(test)} test sentences; {CODE}, and {THIRD} to read.")


## Step 2: before training, and a prediction

First NLLB as it comes: chrF into your language, the direction fine-tuning will train, and into
English, the direction it will not train. chrF counts matching characters, so it moves even
where BLEU is near zero. Then five of your test sentences into your third language, to read
before and after.

Then **predict**, before anything trains: the chrF into your language after 100, 300 and
1,000 steps, and in a sentence, what training only English into your language will do to the
direction into English, and to the third language.

In [ ]:
results_file = OUT_DIR / "t9-results.json"
results = json.loads(results_file.read_text(encoding="utf-8")) if results_file.exists() else {}
if "before" not in results:
    five = list(test.src[:5])
    results["before"] = {"chrf": chrf_both_ways(model), "third": translate(model, five, ENG, THIRD)}
    results_file.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
into_x, into_en = results["before"]["chrf"]
print(f"NLLB as it comes:  chrF into {CODE} {into_x:6.2f}    into English {into_en:6.2f}")


In [ ]:
PREDICTED = {100: None, 300: None, 1000: None}   # <- chrF into your language after each budget
FORGETTING = ""   # <- what training only into your language will do to the other two

assert None not in PREDICTED.values(), "Write a chrF for each budget in PREDICTED, then run this again."
assert FORGETTING.strip(), "Say in FORGETTING what you expect, then run this cell again."
print("Predictions recorded.")


## Step 3: the learning curve

One fine-tuning run, scored three times: after 100 steps, after 300 and after 1,000, each
step one update from 32 pairs. It is one run, not three: each stage resumes from the last
one's checkpoint on Drive, as a run cut off by a dropped session would. The validation loss
it prints at each stage is the same signal read without decoding.

Watch where the curve flattens. The first hundred steps usually move the score most, because
they teach the model the things your pairs all share (the register of the Church's texts, its
names and terms); later steps refine. Where the curve is flat, more steps cost time and buy
nothing, and for a Final Project that is the budget to stop at.

The checkpoints take about 8 GB on your Drive; the training checks for room before it starts.

In [ ]:
from torchlingo.training_checkpoint import TrainingCheckpointer
from torchlingo_harness.nllb import fine_tune

run = TrainingCheckpointer(f"tutorial9-{CODE}", checkpoint_dir=OUT_DIR / "checkpoints")
results.setdefault("curve", {})
for steps in (100, 300, 1000):
    if str(steps) not in results["curve"]:
        state = fine_tune(model, tokenizer, train, val, run, steps=steps, device=device,
                          eval_every=100, seed=SEED)
        results["curve"][str(steps)] = {"chrf": chrf_both_ways(model),
                                       "val_loss": round(state.best_val_loss, 4)}
        results_file.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"{'steps':>6}  {'chrF into ' + CODE:>18}  {'you predicted':>14}  {'validation loss':>16}")
print(f"{0:>6}  {results['before']['chrf'][0]:>18.2f}")
for steps, row in results["curve"].items():
    print(f"{steps:>6}  {row['chrf'][0]:>18.2f}  {PREDICTED[int(steps)]:>14}  {row['val_loss']:>16.4f}")


## Step 4: what it cost

The model was trained only from English into your language, and never reminded of anything
else. A neural network has no separate store for each skill: the same weights serve every
direction and language, so moving them toward one can move them away from the others. This is
**forgetting**. Two checks, after 1,000 steps:

- **into English**, from your language: the direction not trained, scored against your own
  test set's English;
- **into your third language**: five sentences, before and after, for you to read. Is the
  output still in that language, still fluent, still faithful?

A14 trains both directions at once, which is one reason it does: what is trained is kept.

In [ ]:
if "third_after" not in results:
    results["third_after"] = translate(model, list(test.src[:5]), ENG, THIRD)
    results_file.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
before_en, after_en = results["before"]["chrf"][1], results["curve"]["1000"]["chrf"][1]
print(f"chrF into English, the direction not trained: {before_en:.2f} before, {after_en:.2f} after "
      f"({after_en - before_en:+.2f})")
print(f"\nInto {THIRD}, before and after:")
for english, before, after in zip(test.src[:5], results["before"]["third"], results["third_after"]):
    print(f"\n  English: {english}\n  before:  {before}\n  after:   {after}")
print(f"\nYou expected: {FORGETTING.strip()}")


## Step 5: a learning rate set too high

The learning rate sets how far each update moves the weights. Fine-tuning uses a small one,
`1e-4`, so as to nudge what 200 languages taught the model rather than overwrite it. Here is
NLLB as it came, fine-tuned for 100 steps at thirty times that, `3e-3`. Read its validation
loss against Step 3's at 100 steps, and its translations against the reference. A rate set
too high shows as a loss that rises or jumps rather than falls, and as output that repeats
itself, drifts into another language, or collapses to a few common words.

This short run checkpoints to a temporary folder rather than your Drive: it is thrown away.

In [ ]:
del model
torch.cuda.empty_cache() if device.type == "cuda" else None
hot = fresh_nllb()
hot_run = TrainingCheckpointer("tutorial9-too-high", checkpoint_dir=tempfile.mkdtemp())
hot_state = fine_tune(hot, tokenizer, train, val, hot_run, steps=100, device=device,
                      eval_every=50, learning_rate=3e-3, seed=SEED, space_needed_gb=3)
hot_x = compute_chrf(translate(hot, list(test.src), ENG, CODE), list(test.tgt)).score
print(f"\nAfter 100 steps at 3e-3: chrF into {CODE} {hot_x:.2f}, validation loss "
      f"{hot_state.best_val_loss:.4f}")
print(f"After 100 steps at 1e-4: chrF into {CODE} {results['curve']['100']['chrf'][0]:.2f}, "
      f"validation loss {results['curve']['100']['val_loss']:.4f}")
for english, reference, output in zip(test.src[:3], test.tgt[:3],
                                      translate(hot, list(test.src[:3]), ENG, CODE)):
    print(f"\n  English:   {english}\n  reference: {reference}\n  3e-3:      {output}")


## What to take from this

Questions to answer for yourself, from your own numbers:

1. **Where did your curve flatten?** Put your predictions beside the scores. If you were
   fine-tuning for a Final Project, what budget would you choose, and what would make you
   change it: more pairs, a harder language, a deadline?
2. **What did training one direction cost?** How far did chrF into English move, and what did
   you see in the third language? Say why the same weights serving every language makes this
   happen, and how A14's training both ways at once guards against it.
3. **How would you know a learning rate was too high,** from the losses alone, before reading
   any output? And from the output?
4. **For a Final Project that fine-tunes,** which of these three would you report, and how?

Everything here is saved in `MyDrive/CS479/tutorial9/t9-results.json`, so a dropped session
loses nothing: run the setup cells and carry on from the step you were on.

---
## When you finish: disconnect the GPU

**Runtime → Disconnect and delete runtime.** Colab charges compute units for every minute a GPU runtime stays connected, even when nothing is running, so a runtime left open after the work is done uses up your monthly units. Closing the browser tab does not end it right away. Everything this notebook writes is in `MyDrive/CS479/tutorial9`, so disconnecting loses nothing.